# Sandbox Notebook: Pipeline Walkthrough

This notebook is meant to be a simple sandbox for running the **current production modules** in `src/` step by step.

We use it to:
- understand what each stage of the pipeline is doing
- inspect intermediate outputs more easily
- check that the notebook still matches the actual project structure

The idea here is not to rewrite logic in the notebook. It should stay as a light wrapper around the code in `src/`.


## 1. Set Up The Project Imports

This cell finds the repo root, adds it to `sys.path`, and imports the same helpers used in the main pipeline.


In [1]:
import sys
from pathlib import Path

import pandas as pd  # type: ignore
from IPython.display import display
from sklearn.model_selection import train_test_split  # type: ignore

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clean_data import clean_dataframe
from src.evaluate import evaluate_model
from src.features import get_feature_preprocessor
from src.infer import load_inference_model, run_inference
from src.load_data import load_raw_data
from src.logger import configure_logging
from src.main import load_config, resolve_repo_path
from src.train import train_model
from src.utils import save_csv, save_json, save_model
from src.validate import validate_dataframe

PROJECT_ROOT


PosixPath('/Users/nicopbeard/Documents/mlops/mlops-group8')

## 2. Load The Config And Start Logging

This uses the same config file and path logic as the production pipeline.


In [2]:
CONFIG = load_config(PROJECT_ROOT / 'config.yaml')

log_file = resolve_repo_path(PROJECT_ROOT, str(CONFIG['paths']['log_file']))
configure_logging(
    log_level=str(CONFIG.get('logging', {}).get('level', 'INFO')),
    log_file=log_file,
)

CONFIG


{'project': {'name': 'Spotify Sound Archetypes',
  'problem_type': 'regression',
  'target_column': 'popularity'},
 'paths': {'raw_data': 'data/raw/SpotifyAudioFeaturesApril2019.csv',
  'clean_data': 'data/processed/clean.csv',
  'model_path': 'models/model.joblib',
  'report_path': 'reports/predictions.csv',
  'metrics_path': 'reports/metrics.json',
  'run_config_path': 'reports/run_config.json',
  'log_file': 'logs/pipeline.log',
  'artifact_cache_dir': '.artifacts'},
 'train': {'test_size': 0.2,
  'val_size': 0.2,
  'seed': 42,
  'rf_n_estimators': 100,
  'rf_max_depth': 10},
 'features': {'quantile_bin': ['duration_ms', 'tempo'],
  'categorical_onehot': ['key', 'mode'],
  'numeric_passthrough': ['acousticness',
   'danceability',
   'energy',
   'instrumentalness',
   'liveness',
   'loudness',
   'speechiness',
   'valence'],
  'n_bins': 5},
 'logging': {'level': 'INFO', 'format': 'text'},
 'wandb': {'enabled': True,
  'entity': 'nicopbeard-ie-university',
  'project': 'spotify-so

## 3. Resolve The Main Project Paths

Everything below writes to the same configured outputs used by `src.main`.


In [3]:
raw_data_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['raw_data'])
clean_data_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['clean_data'])
model_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['model_path'])
predictions_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['report_path'])
metrics_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['metrics_path'])
run_config_path = resolve_repo_path(PROJECT_ROOT, CONFIG['paths']['run_config_path'])

for output_path in [clean_data_path, model_path, predictions_path, metrics_path, run_config_path, log_file]:
    output_path.parent.mkdir(parents=True, exist_ok=True)

{
    'raw_data_path': str(raw_data_path),
    'clean_data_path': str(clean_data_path),
    'model_path': str(model_path),
    'predictions_path': str(predictions_path),
    'metrics_path': str(metrics_path),
    'run_config_path': str(run_config_path),
    'log_file': str(log_file),
}


{'raw_data_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/data/raw/SpotifyAudioFeaturesApril2019.csv',
 'clean_data_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/data/processed/clean.csv',
 'model_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/models/model.joblib',
 'predictions_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/reports/predictions.csv',
 'metrics_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/reports/metrics.json',
 'run_config_path': '/Users/nicopbeard/Documents/mlops/mlops-group8/reports/run_config.json',
 'log_file': '/Users/nicopbeard/Documents/mlops/mlops-group8/logs/pipeline.log'}

## 4. Load The Raw Data

Use the production data loader and take a quick look at the raw dataset.


In [4]:
df_raw = load_raw_data(raw_data_path)

print(f'Raw shape: {df_raw.shape}')
display(df_raw.head())


2026-03-21 12:39:35 | INFO | src.load_data | Attempting to load raw data from /Users/nicopbeard/Documents/mlops/mlops-group8/data/raw/SpotifyAudioFeaturesApril2019.csv
2026-03-21 12:39:35 | INFO | src.utils | Reading CSV from /Users/nicopbeard/Documents/mlops/mlops-group8/data/raw/SpotifyAudioFeaturesApril2019.csv
2026-03-21 12:39:36 | INFO | src.load_data | Raw data loaded successfully. shape=(130663, 17)
Raw shape: (130663, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


## 5. Clean The Data

Apply the same cleaning logic used in the production pipeline and save the cleaned output.


In [5]:
df_clean = clean_dataframe(df_raw, CONFIG['project']['target_column'])
save_csv(df_clean, clean_data_path)

print(f'Clean shape: {df_clean.shape}')
display(df_clean.head())


2026-03-21 12:39:36 | INFO | src.clean_data | Cleaning dataframe...
2026-03-21 12:39:36 | INFO | src.utils | Saving CSV to /Users/nicopbeard/Documents/mlops/mlops-group8/data/processed/clean.csv
Clean shape: (129858, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


## 6. Validate The Input Contract

Before training, we check that the required columns are there and that the data is in a usable state.


In [6]:
required_columns = (
    CONFIG['features']['quantile_bin']
    + CONFIG['features']['categorical_onehot']
    + CONFIG['features']['numeric_passthrough']
    + [CONFIG['project']['target_column']]
)

validate_dataframe(
    df_clean,
    required_columns,
    target_column=CONFIG['project']['target_column'],
    allow_feature_nulls=True,
)

'Ready for training'


2026-03-21 12:39:37 | INFO | src.validate | Validating data schema...
2026-03-21 12:39:37 | INFO | src.validate | Schema and null-check validation passed.


'Ready for training'

## 7. Split Into Train, Validation, And Test

This follows the same split logic used in `src.main`.


In [7]:
target_column = CONFIG['project']['target_column']
X = df_clean.drop(columns=[target_column])
y = df_clean[target_column]

test_size = CONFIG['train']['test_size']
val_size = CONFIG['train'].get('val_size', 0.2)
seed = CONFIG['train']['seed']

strat = y if CONFIG['project']['problem_type'] == 'classification' else None
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=seed,
    stratify=strat,
)

strat_trainval = y_trainval if CONFIG['project']['problem_type'] == 'classification' else None
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=val_size,
    random_state=seed,
    stratify=strat_trainval,
)

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'rows': [len(X_train), len(X_val), len(X_test)],
    'share': [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
})


,split,rows,share
0,train,83108,0.639991
1,val,20778,0.160006
2,test,25972,0.200003


## 8. Build Features And Train A Validation Model

Here we build the preprocessing pipeline and train the first model on the train split.


In [8]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=CONFIG['features']['quantile_bin'],
    categorical_onehot_cols=CONFIG['features']['categorical_onehot'],
    numeric_passthrough_cols=CONFIG['features']['numeric_passthrough'],
    n_bins=CONFIG['features']['n_bins'],
)

model_pipeline = train_model(
    X_train,
    y_train,
    preprocessor,
    CONFIG['project']['problem_type'],
    train_config=CONFIG.get('train', {}),
)

val_metrics = evaluate_model(
    model_pipeline,
    X_val,
    y_val,
    CONFIG['project']['problem_type'],
)

val_metrics


2026-03-21 12:39:37 | INFO | src.features | Building feature preprocessor recipe...
2026-03-21 12:39:37 | INFO | src.train | Training regression model...
2026-03-21 12:39:45 | INFO | src.evaluate | Evaluating model performance...
2026-03-21 12:39:45 | INFO | src.evaluate | Metrics: rmse=18.237008 mae=14.914996


{'rmse': 18.237008290824583, 'mae': 14.914995926219108}

## 9. Retrain The Final Model And Evaluate It

Like in the main pipeline, we retrain on train plus validation, save the model, and then evaluate on the held-out test set.


In [9]:
final_preprocessor = get_feature_preprocessor(
    quantile_bin_cols=CONFIG['features']['quantile_bin'],
    categorical_onehot_cols=CONFIG['features']['categorical_onehot'],
    numeric_passthrough_cols=CONFIG['features']['numeric_passthrough'],
    n_bins=CONFIG['features']['n_bins'],
)

final_model_pipeline = train_model(
    X_trainval,
    y_trainval,
    final_preprocessor,
    CONFIG['project']['problem_type'],
    train_config=CONFIG.get('train', {}),
)

save_model(final_model_pipeline, model_path)

test_metrics = evaluate_model(
    final_model_pipeline,
    X_test,
    y_test,
    CONFIG['project']['problem_type'],
)

metrics = {'val': val_metrics, 'test': test_metrics}
save_json(metrics, metrics_path)
save_json(CONFIG, run_config_path)

metrics


2026-03-21 12:39:45 | INFO | src.features | Building feature preprocessor recipe...
2026-03-21 12:39:45 | INFO | src.train | Training regression model...
2026-03-21 12:39:55 | INFO | src.utils | Saving model to /Users/nicopbeard/Documents/mlops/mlops-group8/models/model.joblib
2026-03-21 12:39:55 | INFO | src.evaluate | Evaluating model performance...
2026-03-21 12:39:55 | INFO | src.evaluate | Metrics: rmse=18.146484 mae=14.823458
2026-03-21 12:39:55 | INFO | src.utils | Saving JSON to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/metrics.json
2026-03-21 12:39:55 | INFO | src.utils | Saving JSON to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/run_config.json


{'val': {'rmse': 18.237008290824583, 'mae': 14.914995926219108},
 'test': {'rmse': 18.146484220001206, 'mae': 14.823457601968261}}

## 10. Run Inference On Held-Out Rows

Use the production inference helper and save the prediction output.


In [10]:
df_predictions = run_inference(final_model_pipeline, X_test)
save_csv(df_predictions, predictions_path)

display(df_predictions.head())


2026-03-21 12:39:55 | INFO | src.infer | Running inference on new data...
2026-03-21 12:39:56 | INFO | src.infer | Inference complete. Generated 25972 predictions.
2026-03-21 12:39:56 | INFO | src.utils | Saving CSV to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/predictions.csv


,prediction
0,35.803730
1,25.457876
2,25.059417
3,16.995746
4,27.028266


## 11. Optional Check: What Is The API Configured To Load?

This is just a quick way to inspect how the serving side is configured. If `inference.source` is set to `wandb`, the deployed API should be loading the promoted W&B artifact aliased `prod`.


In [11]:
production_summary = {
    'configured_inference_source': CONFIG.get('inference', {}).get('source', 'local'),
    'configured_production_alias': CONFIG.get('wandb', {}).get('production_alias', 'prod'),
    'configured_artifact_name': CONFIG.get('wandb', {}).get('model_artifact_name', ''),
}

production_summary


{'configured_inference_source': 'wandb',
 'configured_production_alias': 'prod',
 'configured_artifact_name': 'spotify-popularity-pipeline'}

## 12. Optional Check: Load The Serving Model The Same Way As The API

If you want, you can also test the serving model loader directly from the notebook. This is optional, but it is useful for checking that the API and the registry setup are aligned.


In [12]:
# Uncomment to test the serving loader end to end.
# serving_model, serving_metadata = load_inference_model(CONFIG, project_root=PROJECT_ROOT)
# serving_metadata
